In [1]:
import pandas as pd, numpy as np

In [2]:
location = "nigeria"
directory = "/snfs1/DATA/DHS_PROG_DHS/NGA/2018/"
results_dir = (
    "../results"
)

### WRA

In [3]:
%%time

wra_columns = {
    "v001": "cluster_number",
    "v002": "household_number",
    "v003": "line_number",
    "v005": "weight",
    "v008": "interview_date",
    "v011": "date_of_birth",
    "v190": "wealth_quintile",
}
wra_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_WN_IAIR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA"),
    columns=wra_columns.keys(),
)

CPU times: user 2.53 s, sys: 452 ms, total: 2.98 s
Wall time: 2.98 s


In [4]:
wra_data = wra_data[wra_columns.keys()].rename(columns=wra_columns)

In [5]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [6]:
wra_data["wealth_quintile"] = recode_wealth_quintile(wra_data.wealth_quintile)

In [7]:
wra_data["weight"] = wra_data.weight / 1_000_000

### Siblings

Sibling survival data appears to only be available as a kind of side table-within-a-table on WRA.

It is labeled "MM" because it is used to calculate maternal mortality (among other things).

In [8]:
respondent_column_names = {
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "v005": "weight",
}

# These are suffixed with an underscore and an integer, e.g. mm1_01
sibling_column_names = {
    # MM1                    Sex of sibling                                  7156    1    N    I   20    0   No   No
    "mm1": "sex",
    # MM2                    Survival status of sibling                      7176    1    N    I   20    0   No   No
    "mm2": "survival_status",
    # MM3                    Sibling's current age                           7196    2    N    I   20    0   No   No
    "mm3": "current_age",
    # MM4                    Sibling's date of birth (CMC)                   7236    4    N    I   20    0   No   No
    "mm4": "date_of_birth",
    # MM8                    Date of death of sibling (CMC)                  7416    4    N    I   20    0   No   No
    "mm8": "date_of_death",
    # MM7                    Sibling's age at death                          7376    2    N    I   20    0   No   No
    "mm7": "age_at_death",
    # MM9                    Sibling's death and pregnancy                   7496    2    N    I   20    0   No   No
    "mm9": "pregnancy_category",
    # MM16                   Sibling's death due to violence or accident     7816    1    N    I   20    0   No   No
    "mm16": "death_violence_or_accident",
}

In [9]:
raw_wra_data = pd.read_stata(directory + "NGA_DHS7_2018_WN_NGIR7AFL_Y2019M11D05.DTA")
sibling_data = raw_wra_data[
    [
        c
        for c in raw_wra_data.columns
        if c in respondent_column_names.keys()
        or c.split("_")[0] in sibling_column_names.keys()
    ]
].copy()
sibling_data

,v005,v008,v190,mm1_01,mm1_02,mm1_03,mm1_04,mm1_05,mm1_06,mm1_07,...,mm16_11,mm16_12,mm16_13,mm16_14,mm16_15,mm16_16,mm16_17,mm16_18,mm16_19,mm16_20
0,1335530,1425,richest,female,female,male,female,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1335530,1425,richest,male,male,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1335530,1425,richest,female,female,female,male,female,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1335530,1425,richest,female,male,male,female,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1335530,1425,richest,male,female,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41816,768129,1426,richer,female,female,male,female,male,male,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41817,768129,1426,richer,female,female,male,male,male,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41818,768129,1426,richer,male,male,male,male,male,male,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41819,768129,1426,richest,female,female,male,male,male,male,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# inspired by https://stackoverflow.com/a/67393747/
sibling_data_reshaped = sibling_data[
    [c for c in sibling_data.columns if c.split("_")[0] in sibling_column_names.keys()]
].copy()
sibling_data_reshaped.columns = sibling_data_reshaped.columns.str.split(
    "_", expand=True
)
sibling_data_reshaped

mm1                                                            ...  \
           01      02      03      04      05    06   07   08   09   10  ...   
0      female  female    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
1        male    male     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
2      female  female  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   
3      female    male    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
4        male  female     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
...       ...     ...     ...     ...     ...   ...  ...  ...  ...  ...  ...   
41816  female  female    male  female    male  male  NaN  NaN  NaN  NaN  ...   
41817  female  female    male    male    male   NaN  NaN  NaN  NaN  NaN  ...   
41818    male    male    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41819  female  female    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41820    male    male  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   

      mm16                                               
        11   12   13   14   15   16   17   18   19   20  
0      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4      NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
...    ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  
41816  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41817  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41818  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41819  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
41820  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[41821 rows x 160 columns]

In [11]:
sibling_data_reshaped[list(respondent_column_names.keys())] = sibling_data[
    list(respondent_column_names.keys())
]
sibling_data_reshaped

mm1                                                            ...  \
           01      02      03      04      05    06   07   08   09   10  ...   
0      female  female    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
1        male    male     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
2      female  female  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   
3      female    male    male  female     NaN   NaN  NaN  NaN  NaN  NaN  ...   
4        male  female     NaN     NaN     NaN   NaN  NaN  NaN  NaN  NaN  ...   
...       ...     ...     ...     ...     ...   ...  ...  ...  ...  ...  ...   
41816  female  female    male  female    male  male  NaN  NaN  NaN  NaN  ...   
41817  female  female    male    male    male   NaN  NaN  NaN  NaN  NaN  ...   
41818    male    male    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41819  female  female    male    male    male  male  NaN  NaN  NaN  NaN  ...   
41820    male    male  female    male  female   NaN  NaN  NaN  NaN  NaN  ...   

      mm16                                v008     v190     v005  
        14   15   16   17   18   19   20                          
0      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
1      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
2      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
3      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
4      NaN  NaN  NaN  NaN  NaN  NaN  NaN  1425  richest  1335530  
...    ...  ...  ...  ...  ...  ...  ...   ...      ...      ...  
41816  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426   richer   768129  
41817  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426   richer   768129  
41818  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426   richer   768129  
41819  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426  richest   768129  
41820  NaN  NaN  NaN  NaN  NaN  NaN  NaN  1426  richest   768129  

[41821 rows x 163 columns]

In [12]:
# Get a row per sibling
sibling_data_reshaped = (
    sibling_data_reshaped.set_index(list(respondent_column_names.keys()))
    .swaplevel(axis=1)
    .stack(0)
    .reset_index()
    .drop(columns=[f"level_{len(respondent_column_names)}"])
)
sibling_data_reshaped

,v008,v190,v005,mm1,mm16,mm2,mm3,mm4,mm7,mm8,mm9
0,1425,richest,1335530,female,NaN,alive,53.0,783.0,NaN,NaN,NaN
1,1425,richest,1335530,female,NaN,alive,50.0,819.0,NaN,NaN,NaN
2,1425,richest,1335530,male,NaN,alive,46.0,867.0,NaN,NaN,NaN
3,1425,richest,1335530,female,NaN,alive,43.0,903.0,NaN,NaN,NaN
4,1425,richest,1335530,male,NaN,alive,24.0,1131.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
219556,1426,richest,768129,male,NaN,alive,38.0,964.0,NaN,NaN,NaN
219557,1426,richest,768129,male,NaN,alive,36.0,988.0,NaN,NaN,NaN
219558,1426,richest,768129,female,NaN,alive,34.0,1012.0,NaN,NaN,NaN
219559,1426,richest,768129,male,NaN,alive,28.0,1084.0,NaN,NaN,NaN


In [13]:
def recode_wealth_quintile(df):
    return df.map(
        {
            "poorest": "lowest",
            "poorer": "second",
            "middle": "middle",
            "richer": "fourth",
            "richest": "highest",
        }
    )

In [14]:
sibling_data = (
    sibling_data_reshaped[
        list(respondent_column_names.keys()) + list(sibling_column_names.keys())
    ]
    .rename(columns=respondent_column_names)
    .rename(columns=sibling_column_names)
)
sibling_data["wealth_quintile"] = recode_wealth_quintile(sibling_data.wealth_quintile)
sibling_data["weight"] = sibling_data.weight / 1_000_000
sibling_data

,interview_date,wealth_quintile,weight,sex,survival_status,current_age,date_of_birth,date_of_death,age_at_death,pregnancy_category,death_violence_or_accident
0,1425,highest,1.335530,female,alive,53.0,783.0,NaN,NaN,NaN,NaN
1,1425,highest,1.335530,female,alive,50.0,819.0,NaN,NaN,NaN,NaN
2,1425,highest,1.335530,male,alive,46.0,867.0,NaN,NaN,NaN,NaN
3,1425,highest,1.335530,female,alive,43.0,903.0,NaN,NaN,NaN,NaN
4,1425,highest,1.335530,male,alive,24.0,1131.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
219556,1426,highest,0.768129,male,alive,38.0,964.0,NaN,NaN,NaN,NaN
219557,1426,highest,0.768129,male,alive,36.0,988.0,NaN,NaN,NaN,NaN
219558,1426,highest,0.768129,female,alive,34.0,1012.0,NaN,NaN,NaN,NaN
219559,1426,highest,0.768129,male,alive,28.0,1084.0,NaN,NaN,NaN,NaN


In [15]:
# "A total of 219,561 siblings were recorded..." (p. 372)
len(sibling_data)

219561

### Births

In [16]:
birth_columns = {
    "v005": "weight",
    "v008": "interview_date",
    "v190": "wealth_quintile",
    "b3": "birth_date",
    "m18": "size_of_child",
    "m19": "birth_weight_kilograms",
    
}
if location == "india":
    birth_columns["s220a"] = "duration_of_pregnancy"
else:
    birth_columns["b20"] = "duration_of_pregnancy"

birth_data = pd.read_stata(
    directory + ("IND_DHS7_2015_2016_BR_IABR74FL_Y2018M12D06.DTA" if location == "india" else "NGA_DHS7_2018_BR_NGBR7AFL_Y2019M11D05.DTA"),
    columns=birth_columns.keys(),
)
birth_data

,v005,v008,v190,b3,m18,m19,b20
0,1335530,1425,richest,1412,average,3500.0,9.0
1,1335530,1425,richest,1292,NaN,NaN,NaN
2,1335530,1425,richest,1206,NaN,NaN,NaN
3,1335530,1425,richest,1416,very small,2000.0,9.0
4,1335530,1425,richest,1361,NaN,NaN,9.0
...,...,...,...,...,...,...,...
127540,768129,1426,richer,1418,larger than average,3200.0,9.0
127541,768129,1426,richest,1384,very large,4100.0,9.0
127542,768129,1426,richest,1366,NaN,NaN,9.0
127543,768129,1426,richest,1355,NaN,NaN,NaN


In [17]:
birth_data = birth_data[birth_columns.keys()].rename(columns=birth_columns)
birth_data["wealth_quintile"] = recode_wealth_quintile(birth_data.wealth_quintile)
birth_data["weight"] = birth_data.weight / 1_000_000
birth_data

,weight,interview_date,wealth_quintile,birth_date,size_of_child,birth_weight_kilograms,duration_of_pregnancy
0,1.335530,1425,highest,1412,average,3500.0,9.0
1,1.335530,1425,highest,1292,NaN,NaN,NaN
2,1.335530,1425,highest,1206,NaN,NaN,NaN
3,1.335530,1425,highest,1416,very small,2000.0,9.0
4,1.335530,1425,highest,1361,NaN,NaN,9.0
...,...,...,...,...,...,...,...
127540,0.768129,1426,fourth,1418,larger than average,3200.0,9.0
127541,0.768129,1426,highest,1384,very large,4100.0,9.0
127542,0.768129,1426,highest,1366,NaN,NaN,9.0
127543,0.768129,1426,highest,1355,NaN,NaN,NaN


### Maternal mortality ratio

#### Maternal mortality rate

In [18]:
sibling_data.survival_status.value_counts()

alive         193315
dead           26226
don't know        20
Name: survival_status, dtype: int64

In [19]:
sibling_data.pregnancy_category.value_counts()

death not related          2974
died during delivery        544
died while pregnant         425
6 weeks after delivery      204
2 months after delivery      43
Name: pregnancy_category, dtype: int64

In [20]:
sibling_data.death_violence_or_accident.value_counts()

no          7147
accident     333
violence     154
Name: death_violence_or_accident, dtype: int64

In [21]:
# https://dhsprogram.com/Data/Guide-to-DHS-Statistics/Adult_Mortality_Rates.htm#Calculation1
sibling_data["exposure_start"] = np.maximum(
    sibling_data.date_of_birth + 12 * 15, sibling_data.interview_date - 84
)  # aka lowlim
# aka upplim
sibling_data["exposure_end"] = np.minimum(
    np.where(
        sibling_data.survival_status == "alive",
        sibling_data.interview_date - 1,
        sibling_data.date_of_death,
    ),
    sibling_data.date_of_birth + 12 * 50 - 1,
)
sibling_data["exposure"] = (
    (sibling_data.exposure_end - sibling_data.exposure_start) + 1
).clip(lower=0)

In [22]:
sibling_data.exposure.value_counts()

84.0    119939
0.0      56888
66.0      7776
42.0      6846
78.0      5633
         ...  
35.0         3
72.0         1
12.0         1
24.0         1
48.0         1
Name: exposure, Length: 84, dtype: int64

In [23]:
sibling_data["adult_death"] = (
    (sibling_data.survival_status == "dead")
    & (sibling_data.date_of_death - sibling_data.date_of_birth >= 15.0 * 12)
    & (sibling_data.date_of_death - sibling_data.date_of_birth < 50.0 * 12)
    & (sibling_data.exposure > 0)
    & (sibling_data.date_of_death >= sibling_data.exposure_start)
    & (sibling_data.date_of_death <= sibling_data.exposure_end)
)

In [24]:
# Matches table 14.2
(
    sibling_data[sibling_data.date_of_birth.notnull()]
    .assign(weighted_exposure=lambda df: df.exposure * df.weight)
    .groupby("sex")
    .weighted_exposure.sum()
    / 12
)

sex
female    480382.344895
male      509840.974981
Name: weighted_exposure, dtype: float64

In [25]:
# Matches table 14.2
sibling_data.assign(
    weighted_adult_dealth=lambda df: df.adult_death * df.weight
).groupby("sex").weighted_adult_dealth.sum()

sex
female    1442.193558
male      1541.674009
Name: weighted_adult_dealth, dtype: float64

In [26]:
sibling_data.death_violence_or_accident.value_counts()

no          7147
accident     333
violence     154
Name: death_violence_or_accident, dtype: int64

In [27]:
sibling_data.pregnancy_category.value_counts()

death not related          2974
died during delivery        544
died while pregnant         425
6 weeks after delivery      204
2 months after delivery      43
Name: pregnancy_category, dtype: int64

In [28]:
female_siblings = sibling_data[sibling_data.sex == "female"].copy()
female_siblings["maternal_death"] = (
    (female_siblings.adult_death)
    & (
        female_siblings.pregnancy_category.isin(
            ["died during delivery", "died while pregnant", "6 weeks after delivery"]
        )
    )
    & (~female_siblings.death_violence_or_accident.isin(["violence", "accident"]))
)

In [29]:
# Table 14.4 reports 451 maternal deaths
(female_siblings.maternal_death * female_siblings.weight).sum()

450.75368699999996

In [30]:
# Table 14.4 reports 480,382
(female_siblings.exposure * female_siblings.weight / 12).sum()

480382.3448947499

In [31]:
def maternal_mortality_rate(df):
    return ((df.maternal_death * df.weight).sum() * 1_000) / (
        (df.exposure * df.weight) / 12
    ).sum()

In [32]:
# "the maternal mortality rate among women age 15-49 is 0.92 deaths per 1,000 woman-years of exposure." (p. 374)
# TODO: These are not age-standardized! We figure the *disparity* probably isn't way off.
# Should standardize according to the approach from https://github.com/LateraOlana/Fertility_SIM_DHS/blob/main/fertility/Latera_Zebb_Coworking.ipynb
maternal_mortality_rate(female_siblings)

0.9383227585076186

In [33]:
maternal_mortality_rates = female_siblings.groupby("wealth_quintile").apply(
    maternal_mortality_rate
)
maternal_mortality_rates

wealth_quintile
lowest     1.460666
second     1.268344
middle     0.856495
fourth     0.787394
highest    0.485920
dtype: float64

#### General fertility rate

In [34]:
fertility_event_data = birth_data.copy()
fertility_event_data["birth_in_period"] = (
    (fertility_event_data.interview_date - fertility_event_data.birth_date) >= 1
) & ((fertility_event_data.interview_date - fertility_event_data.birth_date) <= 36)
fertility_event_data["weighted_birth_in_period"] = (
    fertility_event_data.birth_in_period * fertility_event_data.weight
)

In [35]:
fertility_event_data.weighted_birth_in_period.sum()

20018.820107

In [36]:
fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()

wealth_quintile
lowest     4326.021447
second     4542.606853
middle     4122.133768
fourth     3727.551863
highest    3300.506176
Name: weighted_birth_in_period, dtype: float64

In [37]:
fertility_exposure_data = wra_data.copy()
fertility_exposure_data["exposure_start"] = np.maximum(
    fertility_exposure_data.date_of_birth + 12 * 15,
    fertility_exposure_data.interview_date - 36,
)  # aka lowlim
# aka upplim
fertility_exposure_data["exposure_end"] = np.minimum(
    fertility_exposure_data.interview_date - 1,
    fertility_exposure_data.date_of_birth + 12 * 45,
)
fertility_exposure_data["exposure"] = (
    (fertility_exposure_data.exposure_end - fertility_exposure_data.exposure_start) + 1
).clip(lower=0)
fertility_exposure_data["weighted_exposure"] = (
    fertility_exposure_data.exposure * fertility_exposure_data.weight
)

In [38]:
# Within rounding error of value reported in Table 5.1
fertility_event_data.weighted_birth_in_period.sum() * 1_000 / (
    fertility_exposure_data.weighted_exposure.sum() / 12
)

181.80045597312596

In [39]:
gfr_by_wealth = (
    fertility_event_data.groupby("wealth_quintile").weighted_birth_in_period.sum()
    * 1_000
    / (fertility_exposure_data.groupby("wealth_quintile").weighted_exposure.sum() / 12)
)
gfr_by_wealth

wealth_quintile
lowest     228.551509
second     214.895278
middle     191.264704
fourth     158.111396
highest    132.443590
dtype: float64

In [40]:
maternal_disorders_incidence_disparities = (
    (maternal_mortality_rates / gfr_by_wealth)
    .rename("value")
    .rename_axis("wealth_quintile")
    .reset_index()
)
maternal_disorders_incidence_disparities.insert(0, "sex", "Female")
maternal_disorders_incidence_disparities

,sex,wealth_quintile,value
0,Female,lowest,0.006391
1,Female,second,0.005902
2,Female,middle,0.004478
3,Female,fourth,0.004980
4,Female,highest,0.003669


In [41]:
maternal_disorders_incidence_disparities.to_csv(
    f"{results_dir}/maternal_disorders_incidence_disparities/nigeria.csv",
    index=False,
)